# ffn_cellsim Phase-C on a Google Colab GPU

Runs the Warp DCM spheroid engine on a **Colab GPU**, in parallel with the gbook A5000. Each
Colab GPU runs **one** config — duplicate this notebook across GPUs (or use the param cell) to
sweep, e.g. `--accel-dt` (the stiff polarized+substrate solver work) or IPC-vs-penalty contact.

**Setup**: Runtime ▸ Change runtime type ▸ **GPU** (T4/L4/A100). Then run the cells top to bottom.

**Auth**: the session branch `h7/m1-ipc-contact` must be on GitHub (Claude/PI pushes it). Paste a
GitHub read token in `TOKEN` below, or make the repo public.

In [ ]:
# 1. confirm the Colab GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# 2. clone the repo + checkout the session branch
import os
TOKEN  = ''                        # <- paste a GitHub PAT (repo:read), or leave '' if the repo is public
BRANCH = 'h7/m1-ipc-contact'       # the session branch (must be pushed to GitHub)
auth = f'{TOKEN}@' if TOKEN else ''
url  = f'https://{auth}github.com/sungwook9997/ffn_cellsim.git'
if not os.path.isdir('ffn_cellsim'):
    !git clone --branch {BRANCH} --depth 1 {url} ffn_cellsim
%cd ffn_cellsim
!git log --oneline -1

In [ ]:
# 3. install deps (Colab has numpy/scipy/matplotlib; add Warp)
!pip -q install warp-lang==1.14.0
import warp as wp; wp.init()   # JIT-compiles the CUDA kernels on first use

In [ ]:
# 4a. SELF-TEST — the working aggregation (proves the Colab GPU path runs end-to-end).
#     ~12 min on a T4; produces a clean round all-touching spheroid (pen~0.12).
import os, subprocess
os.environ['PYTHONPATH'] = '.'
os.makedirs('/content/out', exist_ok=True)
subprocess.run(['python','-m','ffn_sim.scripts.run_spread_overnight',
                '--n','100','--steps','8000','--settle','6000','--ipc','--no-bundle',
                '--device','cuda:0','--tag','colab_agg','--out','/content/out'])

In [ ]:
# 4b. SWEEP cell — set ONE value per GPU and duplicate the notebook to parallelize.
#     The PI-fork option (c): find a stable+fast accel_dt for the stiff polarized spreading
#     (8e-4 diverges/crawls; try 8e-5, 8e-6). Or flip --polarize/--ipc/--k-vol for other A/Bs.
import os, subprocess
os.environ['PYTHONPATH'] = '.'
ACCEL_DT = '8e-5'   # <- sweep this across Colab GPUs
subprocess.run(['python','-m','ffn_sim.scripts.run_spread_overnight',
                '--n','100','--steps','20000','--ipc','--k-vol','1e3','--polarize',
                '--gamma-surf','1e-4','--well','--no-bundle','--cfl-limit','0.3',
                '--accel-dt',ACCEL_DT,'--device','cuda:0',
                '--tag',f'colab_pol_accel{ACCEL_DT}','--out','/content/out'])

In [ ]:
# 5. visualize (nucleus z-montage) + download the npz/json back for the PI / Syncthing
import glob
from ffn_sim.scripts.spheroid_nucleus_viz import render_montage
for npz in glob.glob('/content/out/*.npz'):
    render_montage(npz)
from google.colab import files
for f in glob.glob('/content/out/*.json') + glob.glob('/content/out/figs/*.png'):
    files.download(f)